# Market Impact in Volume-Time — Temporal Impact Dynamics

**Complementary to 104 (square-root law cross-section).**

### What 104 does vs what this notebook does

| | 104 (sqrt law) | 105 (this notebook) |
|---|---|---|
| **Varies** | Q — how much volume is injected | L — how fast the same volume is injected |
| **Time axis** | None — one point per iteration | Normalized u ∈ [0, 1+γ] — full time series |
| **Impact** | VWAP cumulative (endpoint) | Midprice instantaneous (at each step) |
| **Output** | Single number: β ≈ 0.55 | Curve shape: I(u) — buildup, peak, decay |
| **Question** | "How does impact scale with volume?" | "How does execution speed affect impact dynamics?" |

### Framework

We define **normalized time** u = (t − s) / L, where:
- **s** = message index of the first aggressive order injection
- **L** = distance (in messages) between first and last injection: `aggr[-1] - aggr[0]`
- **u ∈ [0, 1]**: execution phase — aggressive orders are being injected between metablocks
- **u > 1**: relaxation phase — model generates freely, no more injections
- **I(u) = midprice(t) − midprice(s−1)**: displacement relative to the last pre-execution midprice

The combined impact is **(buy − sell) / 2** — same anti-drift technique as 103/104. Buy scenarios push price up, sell push price down. Averaging cancels any directional bias.

### Experiment grid: same Q, different L

The c10x design has groups where **total injected volume Q = i × 75 is fixed**, but **execution duration L varies** (because metablock size mb differs):

| Group | Q (shares) | i | Experiments | L (msgs) | mb |
|-------|-----------|---|-------------|----------|-----|
| **Q=225** | 3×75 | 3 | mb5, mb10, mb15 | 12, 22, 32 | 5, 10, 15 |
| **Q=150** | 2×75 | 2 | mb10, mb15, mb20 | 11, 16, 21 | 10, 15, 20 |

Within each group, **larger mb → larger L → slower execution of the same volume Q**. The question is: does the model respond differently to fast vs slow execution?

**Note on mb=20**: Experiments with mb=20 showed pathological behavior in 103 (near-zero decay). They are included for completeness but should be interpreted with caution.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import re

## Section 0: Data Loading

Reuses the same data loaders as 103/104. Each experiment folder contains:
- **data_cond/**: 500 historical context messages (real LOBSTER data)
- **data_gen/**: generated continuation — model's response to injected aggressive orders
- **aggressive_indices.csv**: positions (within gen sequence) where aggressive orders were injected

For each sample we concatenate `[cond_book, gen_book]` into one trajectory and track the **junction** (boundary between real and generated data).

In [ ]:
MAX_SAMPLES = 2048
MIDPRICE_MAX = 2_000_000
CONTEXT = 500
TICK_SIZE = 100
SHARES_PER_INSERTION = 75

# Auto-detect: Docker (/app) vs host
_BASE = Path("/app/output/evalsequences/aggressive_scenario")
if not _BASE.exists():
    _BASE = Path("/scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences/aggressive_scenario")

BUY_PATH = _BASE / "context_500_buy"
SELL_PATH = _BASE / "context_500_sell"

print(f"Using base: {_BASE}")
print(f"BUY_PATH exists: {BUY_PATH.exists()}")

FOLDERS = [
    "i3_c30_mb5_cntxt33%",
    "i5_c50_mb5_cntxt55%",
    "i9_c90_mb5_cntxt99%",
    "i2_c20_mb10_cntxt44%",
    "i3_c30_mb10_cntxt66%",
    "i4_c40_mb10_cntxt88%",
    "i2_c20_mb15_cntxt66%",
    "i3_c30_mb15_cntxt99%",
    "i1_c10_mb20_cntxt44%",
    "i2_c20_mb20_cntxt88%",
]

_SDM_PATH = Path("/app/lob_impact/sample_day_map.csv")
if not _SDM_PATH.exists():
    _SDM_PATH = Path("/scratch/local/homes/80/georgenigm/LOBS5/lob_impact/sample_day_map.csv")
SAMPLE_DAY_MAP = pd.read_csv(_SDM_PATH)
print(f"Loaded sample_day_map: {len(SAMPLE_DAY_MAP)} rows")

# Q-groups: same total volume Q = i * 75, different execution duration L
Q_GROUPS = {
    225: [  # i=3
        'i3_c30_mb5_cntxt33%',
        'i3_c30_mb10_cntxt66%',
        'i3_c30_mb15_cntxt99%',
    ],
    150: [  # i=2
        'i2_c20_mb10_cntxt44%',
        'i2_c20_mb15_cntxt66%',
        'i2_c20_mb20_cntxt88%',
    ],
}

# Colors by metablock size
MB_COLORS = {
    5:  ('#e41a1c', 'rgba(228,26,28,0.12)'),
    10: ('#377eb8', 'rgba(55,126,184,0.12)'),
    15: ('#4daf4a', 'rgba(77,175,74,0.12)'),
    20: ('#984ea3', 'rgba(152,78,163,0.12)'),
}

In [ ]:
def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / "data_cond"
    pattern = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    samples = []
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        match = pattern.match(f.name)
        if match:
            samples.append((match.group(1), match.group(2), int(match.group(3))))
    if not samples:
        raise ValueError(f"No orderbook files found in {cond_dir}")
    samples.sort()
    if max_samples is not None and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def is_midprice_outlier(book_array, max_midprice):
    midprice = (book_array[:, 0] + book_array[:, 2]) / 2
    return np.any(midprice > max_midprice) or np.any(midprice <= 0)


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    samples = discover_data_params(data_path, max_samples=max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    n_outliers = 0
    for ticker, date, sid in samples:
        cond_book_path = data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv"
        gen_book_path = data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv"
        gen_msg_path = data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv"
        if not gen_book_path.exists():
            continue
        cond_book = np.loadtxt(cond_book_path, delimiter=',')
        gen_book = np.loadtxt(gen_book_path, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice is not None and is_midprice_outlier(full_book, max_midprice):
            n_outliers += 1
            continue
        gen_msg = np.loadtxt(gen_msg_path, delimiter=',')
        cond_msg_path = data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv"
        cond_msg = np.loadtxt(cond_msg_path, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key] = np.vstack([cond_msg, gen_msg])
    if not gen_books:
        raise ValueError(f"No complete sample pairs found in {data_path}")
    if n_outliers > 0:
        print(f"  filtered {n_outliers} outlier samples (midprice > {max_midprice})")
    return gen_books, gen_msgs, cond_lens


def load_all():
    all_data = {}
    for folder in FOLDERS:
        buy_path = BUY_PATH / folder
        sell_path = SELL_PATH / folder
        if not buy_path.exists() or not sell_path.exists():
            print(f"SKIP: {folder} (missing buy or sell)")
            continue
        try:
            buy_books, buy_msgs, buy_cond = load_folder_data(
                buy_path, max_samples=MAX_SAMPLES, max_midprice=MIDPRICE_MAX)
            sell_books, sell_msgs, sell_cond = load_folder_data(
                sell_path, max_samples=MAX_SAMPLES, max_midprice=MIDPRICE_MAX)
            all_data[folder] = {
                'buy': {'books': buy_books, 'msgs': buy_msgs, 'cond_lens': buy_cond},
                'sell': {'books': sell_books, 'msgs': sell_msgs, 'cond_lens': sell_cond},
            }
            print(f"OK: {folder} (buy={len(buy_books)}, sell={len(sell_books)} samples)")
        except Exception as e:
            print(f"ERROR: {folder} - {e}")
    return all_data

In [ ]:
data = load_all()

### Core computation: volume-time curves

For each experiment, the pipeline is:

1. **Load aggressive_indices** — e.g. `[5, 11, 17]` for i3_mb5. These are gen-relative positions. Spacing = mb+1 (each metablock of mb generated messages + 1 injected aggressive order).

2. **Define L** = `aggr[-1] - aggr[0]` = actual execution duration in messages. For i3_mb5: L = 17−5 = 12.

3. **For each sample** (2048 buy + 2048 sell):
   - Compute full midprice trajectory: `(best_ask + best_bid) / 2`
   - Reference price: `p_ref = midprice[junction + aggr[0] - 1]` (last pre-execution)
   - Raw impact at each step: `impact(t) = midprice(t) - p_ref`
   - Convert to normalized time: `u(t) = (t - s) / L`
   - Interpolate onto common u-grid (500 points) for cross-sample averaging

4. **Combine buy and sell**: `I(u) = (buy_mean(u) - sell_mean(u)) / 2`

Two versions:
- **Raw** (`compute_volume_time_curves`): I(u) in price units (cents)
- **Sigma-normalized** (`compute_volume_time_curves_sigma_norm`): I(u) / (p_ref × σ_day) — dimensionless, normalized by Parkinson daily volatility. Used for cross-experiment comparison in Section 5.

In [ ]:
def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2


def parse_folder_params(folder_name):
    match = re.match(r'i(\d+)_c(\d+)_mb(\d+)_cntxt(.+)', folder_name)
    if match:
        return int(match.group(1)), int(match.group(2)), int(match.group(3))
    return None, None, None


def load_aggressive_indices(data_path):
    aggr_file = data_path / 'aggressive_indices.csv'
    if not aggr_file.exists():
        return np.array([], dtype=int)
    indices = np.loadtxt(aggr_file, dtype=int)
    if indices.ndim == 0:
        indices = np.array([int(indices)])
    return indices


def compute_volume_time_curves(buy_data, sell_data, folder, aggr_indices_gen,
                                u_max=3.0, n_u_points=300):
    """Compute combined impact curve in normalized volume-time coordinates.

    u = (t - s) / L where s = first aggressive index (absolute), L = last - first.
    Impact I(u) = midprice(t) - midprice(s-1) in price units.
    Combined = (buy_impact - sell_impact) / 2.
    """
    i_val, c_val, mb_val = parse_folder_params(folder)

    if len(aggr_indices_gen) < 2:
        return None

    s_gen = int(aggr_indices_gen[0])
    e_gen = int(aggr_indices_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None

    buy_books = buy_data['books']
    sell_books = sell_data['books']

    min_len = min(
        min(b.shape[0] for b in buy_books.values()),
        min(b.shape[0] for b in sell_books.values())
    )

    # Cap u_grid by actual data extent
    junction_ex = list(buy_data['cond_lens'].values())[0]
    u_data_max = (min_len - 1 - junction_ex - s_gen) / L
    u_cap = min(u_max, u_data_max)
    u_grid = np.linspace(0, u_cap, n_u_points)

    def process_side(books, cond_lens):
        impacts = []
        for sid, book_arr in books.items():
            junction = cond_lens[sid]
            s_abs = junction + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            midprice = compute_midprice(book_arr[:min_len])
            p_ref = midprice[s_abs - 1]
            n_steps = min_len - s_abs
            impact_raw = midprice[s_abs:min_len] - p_ref
            u_raw = np.arange(n_steps) / L
            impact_interp = np.interp(u_grid, u_raw, impact_raw)
            impacts.append(impact_interp)
        return np.array(impacts) if impacts else None

    buy_impacts = process_side(buy_books, buy_data['cond_lens'])
    sell_impacts = process_side(sell_books, sell_data['cond_lens'])

    if buy_impacts is None or sell_impacts is None:
        return None

    buy_mean = np.mean(buy_impacts, axis=0)
    sell_mean = np.mean(sell_impacts, axis=0)
    combined_mean = (buy_mean - sell_mean) / 2
    combined_std = np.sqrt(np.std(buy_impacts, axis=0)**2 + np.std(sell_impacts, axis=0)**2) / 2

    return {
        'u_grid': u_grid,
        'combined_mean': combined_mean,
        'combined_std': combined_std,
        'buy_mean': buy_mean,
        'sell_mean': sell_mean,
        'L': L,
        's_gen': s_gen,
        'e_gen': e_gen,
        'i': i_val,
        'c': c_val,
        'mb': mb_val,
        'Q': i_val * SHARES_PER_INSERTION,
        'gamma': c_val / i_val,
        'n_buy': buy_impacts.shape[0],
        'n_sell': sell_impacts.shape[0],
    }


def compute_volume_time_curves_sigma_norm(buy_data, sell_data, folder, aggr_indices_gen,
                                           u_max=3.0, n_u_points=300):
    """Same as compute_volume_time_curves but normalizes each sample's impact
    by p_ref * sigma_day. Returns impact in units of daily volatility."""
    i_val, c_val, mb_val = parse_folder_params(folder)

    if len(aggr_indices_gen) < 2:
        return None

    s_gen = int(aggr_indices_gen[0])
    e_gen = int(aggr_indices_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None

    buy_books = buy_data['books']
    sell_books = sell_data['books']

    min_len = min(
        min(b.shape[0] for b in buy_books.values()),
        min(b.shape[0] for b in sell_books.values())
    )

    junction_ex = list(buy_data['cond_lens'].values())[0]
    u_data_max = (min_len - 1 - junction_ex - s_gen) / L
    u_cap = min(u_max, u_data_max)
    u_grid = np.linspace(0, u_cap, n_u_points)

    def process_side(books, cond_lens):
        impacts = []
        for sid, book_arr in books.items():
            sample_id = sid[1]
            day_row = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
            if day_row.empty:
                continue
            H = float(day_row.iloc[0]['highest_price']) / TICK_SIZE
            L_price = float(day_row.iloc[0]['lowest_price']) / TICK_SIZE
            if H <= L_price or L_price <= 0:
                continue
            sigma_day = np.log(H / L_price) / 0.8325546
            if sigma_day <= 0:
                continue

            junction = cond_lens[sid]
            s_abs = junction + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            midprice = compute_midprice(book_arr[:min_len])
            p_ref = midprice[s_abs - 1]
            if p_ref <= 0:
                continue
            n_steps = min_len - s_abs
            # Normalize: return / sigma_day
            impact_raw = (midprice[s_abs:min_len] - p_ref) / (p_ref * sigma_day)
            u_raw = np.arange(n_steps) / L
            impact_interp = np.interp(u_grid, u_raw, impact_raw)
            impacts.append(impact_interp)
        return np.array(impacts) if impacts else None

    buy_impacts = process_side(buy_books, buy_data['cond_lens'])
    sell_impacts = process_side(sell_books, sell_data['cond_lens'])

    if buy_impacts is None or sell_impacts is None:
        return None

    buy_mean = np.mean(buy_impacts, axis=0)
    sell_mean = np.mean(sell_impacts, axis=0)
    combined_mean = (buy_mean - sell_mean) / 2
    combined_std = np.sqrt(np.std(buy_impacts, axis=0)**2 + np.std(sell_impacts, axis=0)**2) / 2

    return {
        'u_grid': u_grid,
        'combined_mean': combined_mean,
        'combined_std': combined_std,
        'L': L,
        'i': i_val,
        'c': c_val,
        'mb': mb_val,
        'Q': i_val * SHARES_PER_INSERTION,
        'gamma': c_val / i_val,
        'n_buy': buy_impacts.shape[0],
        'n_sell': sell_impacts.shape[0],
    }

In [ ]:
# Compute volume-time curves for all experiments (both raw and sigma-normalized)
all_curves = {}
all_curves_norm = {}

for folder in FOLDERS:
    if folder not in data:
        continue
    aggr = load_aggressive_indices(BUY_PATH / folder)
    if len(aggr) < 2:
        print(f"SKIP {folder}: < 2 aggressive indices (i=1)")
        continue

    curve = compute_volume_time_curves(
        data[folder]['buy'], data[folder]['sell'], folder, aggr,
        u_max=11.0, n_u_points=500)
    if curve:
        all_curves[folder] = curve
        print(f"OK: {folder} | Q={curve['Q']} shares, L={curve['L']} msgs, "
              f"mb={curve['mb']}, gamma={curve['gamma']:.0f}, "
              f"n=({curve['n_buy']}+{curve['n_sell']})")

    curve_norm = compute_volume_time_curves_sigma_norm(
        data[folder]['buy'], data[folder]['sell'], folder, aggr,
        u_max=11.0, n_u_points=500)
    if curve_norm:
        all_curves_norm[folder] = curve_norm

print(f"\nTotal curves: {len(all_curves)} raw, {len(all_curves_norm)} sigma-normalized")

**Note:** Experiments with i=1 (single insertion) are skipped because L = 0 — there is no "execution duration" to normalize by. These include `i1_c10_mb20`.

The output for each experiment includes:
- `u_grid`: normalized time array, u ∈ [0, ~11] (covering execution + γ=10 relaxation)
- `combined_mean`: average combined impact at each u
- `L`: actual execution duration in messages
- `Q`: total injected volume in shares

## Section 1: Normalized Impact Curves I(u; Q, L)

This is the central plot of the notebook. For each Q-group, we overlay 3 curves (one per execution speed L) on the same axes.

**How to read the plot:**
- **x-axis: u** — normalized time. u=0 is the first aggressive injection, u=1 is the last injection, u>1 is free relaxation.
- **y-axis: I(u)** — combined (buy−sell)/2 midprice displacement in price units (cents for GOOG).
- The **shading** shows ±1 std across ~2048 buy + ~2048 sell samples.

**What to look for:**
- **[0, 1] interval**: impact should build up as aggressive orders are injected. Faster execution (smaller L, fewer msgs between injections) should produce steeper rise.
- **u=1 (vertical dashed line)**: peak impact — execution ends here. This is the moment right after the last aggressive order.
- **u>1**: relaxation — the model generates freely, no more injections. Impact should partially decay ("market resilience").
- **Comparing curves within a Q-group**: all 3 curves inject the SAME total volume Q. The difference is purely execution speed. Slower execution (larger L) is expected to produce **lower peak** (the market has more time to absorb the impact) and a **different decay shape**.

Two views: focused (u ≤ 3, to see the interesting part clearly) and full extent (u ≤ 11, showing the full γ=10 relaxation).

In [ ]:
# Focused view: u in [0, 3] (execution + 2x relaxation)
U_SHOW = 3.0

fig = make_subplots(
    rows=1, cols=len(Q_GROUPS),
    subplot_titles=[f'Q = {Q} shares (i={Q // SHARES_PER_INSERTION})'
                    for Q in sorted(Q_GROUPS.keys(), reverse=True)],
    horizontal_spacing=0.10,
)

for col_idx, Q in enumerate(sorted(Q_GROUPS.keys(), reverse=True), 1):
    folders = Q_GROUPS[Q]
    for folder in folders:
        if folder not in all_curves:
            continue
        c = all_curves[folder]
        u = c['u_grid']
        mask = u <= U_SHOW
        u_show = u[mask]
        mean_show = c['combined_mean'][mask]
        std_show = c['combined_std'][mask]

        line_color, fill_color = MB_COLORS[c['mb']]
        label = f"mb={c['mb']} (L={c['L']})"

        # Std band
        fig.add_trace(go.Scatter(
            x=np.concatenate([u_show, u_show[::-1]]),
            y=np.concatenate([mean_show + std_show, (mean_show - std_show)[::-1]]),
            fill='toself', fillcolor=fill_color,
            line=dict(color='rgba(255,255,255,0)'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=col_idx)

        # Mean line
        fig.add_trace(go.Scatter(
            x=u_show, y=mean_show,
            mode='lines', line=dict(color=line_color, width=2.5),
            name=label, legendgroup=label, showlegend=True,
        ), row=1, col=col_idx)

    # Vertical line at u=1 (end of execution)
    fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1.5,
                  annotation_text='u=1 (exec end)', annotation_position='top',
                  row=1, col=col_idx)
    fig.add_hline(y=0, line_dash='dot', line_color='gray', line_width=1, row=1, col=col_idx)

fig.update_xaxes(title_text='Normalized time u = (t-s)/L', row=1, col=1)
fig.update_yaxes(title_text='Combined impact I(u) (price units)', row=1, col=1)
for col_idx in range(2, len(Q_GROUPS) + 1):
    fig.update_xaxes(title_text='u', row=1, col=col_idx)

fig.update_layout(
    title_text='<b>Section 1: Impact Curves in Volume-Time</b> | u in [0, 3] | Execution = [0, 1]',
    width=1400, height=500, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()

**Full extent view** below shows the complete relaxation (γ=10 means 10× execution duration of cooling). Most of the decay happens in the first u ∈ [1, 3]; the rest is a long plateau. This is consistent with the "fast initial decay + permanent component" pattern from 103.

In [ ]:
# Full extent: u in [0, 11] (execution + full gamma=10 relaxation)
U_FULL = 11.0

fig = make_subplots(
    rows=1, cols=len(Q_GROUPS),
    subplot_titles=[f'Q = {Q} shares (i={Q // SHARES_PER_INSERTION})'
                    for Q in sorted(Q_GROUPS.keys(), reverse=True)],
    horizontal_spacing=0.10,
)

for col_idx, Q in enumerate(sorted(Q_GROUPS.keys(), reverse=True), 1):
    folders = Q_GROUPS[Q]
    for folder in folders:
        if folder not in all_curves:
            continue
        c = all_curves[folder]
        u = c['u_grid']
        mask = u <= U_FULL
        u_show = u[mask]
        mean_show = c['combined_mean'][mask]

        line_color, _ = MB_COLORS[c['mb']]
        label = f"mb={c['mb']} (L={c['L']})"

        fig.add_trace(go.Scatter(
            x=u_show, y=mean_show,
            mode='lines', line=dict(color=line_color, width=2),
            name=label, legendgroup=label, showlegend=True,
        ), row=1, col=col_idx)

    fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1.5,
                  annotation_text='u=1', annotation_position='top',
                  row=1, col=col_idx)
    fig.add_hline(y=0, line_dash='dot', line_color='gray', line_width=1, row=1, col=col_idx)

fig.update_xaxes(title_text='u', row=1, col=1)
fig.update_yaxes(title_text='I(u) (price units)', row=1, col=1)

fig.update_layout(
    title_text='<b>Section 1b: Full Extent</b> | u in [0, 11] | gamma=10',
    width=1400, height=450, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()

## Section 2: Peak Impact vs Execution Speed

**I_peak = I(u=1)** — the midprice displacement at the moment the last aggressive order is injected.

**Why this matters:** In real trading, the execution algorithm controls how fast to trade. The trade-off is:
- **Fast execution** (small L): get done quickly, but the concentrated order flow "leaks information" to the market → higher peak impact → worse execution price.
- **Slow execution** (large L): spread orders over more messages, giving the book time to replenish between injections → lower peak impact → better execution price, but higher exposure to price risk.

**Left plot**: I_peak vs L at fixed Q. Expected pattern: **decreasing** — slower execution → lower peak.

**Right plot**: I_peak vs Q/L (execution intensity = shares per message). This normalizes for Q differences and puts both groups on a comparable axis. Expected pattern: **increasing** — more aggressive trading → higher impact.

In [ ]:
# Extract peak impact I(u=1) and other metrics
peak_rows = []

for folder, c in all_curves.items():
    u = c['u_grid']
    mean = c['combined_mean']

    # I(u=1): interpolate at u=1
    I_peak = float(np.interp(1.0, u, mean))

    # I_final: value at end of data (u ~ gamma)
    I_final = float(mean[-1])

    # Actual max of the curve
    I_max = float(np.max(mean))
    u_max_val = float(u[np.argmax(mean)])

    peak_rows.append({
        'folder': folder,
        'Q': c['Q'],
        'i': c['i'],
        'mb': c['mb'],
        'L': c['L'],
        'gamma': c['gamma'],
        'I_peak_u1': I_peak,
        'I_max': I_max,
        'u_at_max': u_max_val,
        'I_final': I_final,
        'n_samples': c['n_buy'] + c['n_sell'],
    })

peak_df = pd.DataFrame(peak_rows).sort_values(['Q', 'L'], ascending=[False, True]).reset_index(drop=True)

print("Peak Impact Summary:")
print(peak_df.to_string(index=False))

# Highlight Q-group members
for Q, folders in Q_GROUPS.items():
    sub = peak_df[peak_df['folder'].isin(folders)].sort_values('L')
    if len(sub) > 1:
        L_vals = sub['L'].values
        I_vals = sub['I_peak_u1'].values
        decreasing = all(I_vals[i] >= I_vals[i+1] for i in range(len(I_vals)-1))
        print(f"\nQ={Q}: L={list(L_vals)}, I_peak={[f'{v:.2f}' for v in I_vals]}")
        print(f"  Peak decreasing with L: {decreasing}")

In [ ]:
# Plot: I_peak vs L for each Q-group
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['I_peak(u=1) vs L', 'I_peak vs Execution Intensity Q/L'],
    horizontal_spacing=0.12,
)

q_colors = {225: '#e41a1c', 150: '#377eb8'}

for Q, folders in Q_GROUPS.items():
    sub = peak_df[peak_df['folder'].isin(folders)].sort_values('L')
    if sub.empty:
        continue
    color = q_colors.get(Q, 'gray')

    # Left: I_peak vs L
    fig.add_trace(go.Scatter(
        x=sub['L'], y=sub['I_peak_u1'],
        mode='lines+markers+text',
        marker=dict(size=10, color=color),
        line=dict(color=color, width=2),
        text=[f'mb={mb}' for mb in sub['mb']],
        textposition='top center',
        name=f'Q={Q}', legendgroup=f'Q={Q}',
    ), row=1, col=1)

    # Right: I_peak vs Q/L
    fig.add_trace(go.Scatter(
        x=sub['Q'] / sub['L'], y=sub['I_peak_u1'],
        mode='lines+markers+text',
        marker=dict(size=10, color=color),
        line=dict(color=color, width=2),
        text=[f'mb={mb}' for mb in sub['mb']],
        textposition='top center',
        name=f'Q={Q}', legendgroup=f'Q={Q}', showlegend=False,
    ), row=1, col=2)

fig.update_xaxes(title_text='L (execution duration, msgs)', row=1, col=1)
fig.update_yaxes(title_text='I_peak = I(u=1) (price units)', row=1, col=1)
fig.update_xaxes(title_text='Q/L (shares per msg)', row=1, col=2)
fig.update_yaxes(title_text='I_peak (price units)', row=1, col=2)

fig.update_layout(
    title_text='<b>Section 2: Peak Impact vs Execution Speed</b>',
    width=1200, height=450, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()

## Section 3: Relaxation Ratio — How Much Impact Persists?

After the last aggressive order (u=1), the model generates freely. The impact partially decays — the order book rebuilds, new limit orders fill the gaps. But some fraction of the impact is **permanent** (the "information content" of the trade).

**Relaxation ratio** = I_final / I_peak = what fraction of peak impact remains at the end.

**Theory (Bouchaud et al.):** In real markets, roughly **2/3 of peak impact persists** — the rest is transient. This "2/3 rule" is a well-known empirical regularity.

**What we check:**
- `ratio_u2` = I(u=2) / I(u=1) — how much remains after 1× execution duration of relaxation
- `ratio_final` = I(end) / I(u=1) — how much remains at the very end (u ≈ 11)
- Is the ratio **stable across different L** within the same Q-group? If yes, the decay dynamics are universal.

**Green band** [0.5, 0.9] on the plots is a "reasonable range". Values outside this range may indicate pathological behavior (too much decay → impact doesn't persist; too little decay → book doesn't recover).

In [ ]:
# Compute relaxation ratios at multiple u-points
relax_rows = []

for folder, c in all_curves.items():
    u = c['u_grid']
    mean = c['combined_mean']

    I_peak = float(np.interp(1.0, u, mean))
    if abs(I_peak) < 1e-12:
        continue

    # Relaxation at u=2 (1x L after execution)
    I_u2 = float(np.interp(2.0, u, mean)) if u[-1] >= 2.0 else np.nan
    # Relaxation at u=3 (2x L after execution)
    I_u3 = float(np.interp(3.0, u, mean)) if u[-1] >= 3.0 else np.nan
    # Final value
    I_final = float(mean[-1])
    u_final = float(u[-1])

    relax_rows.append({
        'folder': folder,
        'Q': c['Q'],
        'mb': c['mb'],
        'L': c['L'],
        'I_peak': I_peak,
        'I_u2': I_u2,
        'I_u3': I_u3,
        'I_final': I_final,
        'u_final': u_final,
        'ratio_u2': I_u2 / I_peak if not np.isnan(I_u2) else np.nan,
        'ratio_u3': I_u3 / I_peak if not np.isnan(I_u3) else np.nan,
        'ratio_final': I_final / I_peak,
    })

relax_df = pd.DataFrame(relax_rows).sort_values(['Q', 'L'], ascending=[False, True]).reset_index(drop=True)

print("Relaxation Ratios:")
print(relax_df[['folder', 'Q', 'mb', 'L', 'I_peak', 'I_u2', 'I_u3', 'I_final',
                'ratio_u2', 'ratio_u3', 'ratio_final']].to_string(index=False))

# Check stability within Q-groups
for Q, folders in Q_GROUPS.items():
    sub = relax_df[relax_df['folder'].isin(folders)]
    if sub.empty:
        continue
    ratios = sub['ratio_u2'].dropna()
    if len(ratios) > 1:
        print(f"\nQ={Q}: ratio_u2 = {[f'{v:.3f}' for v in ratios]}, "
              f"std={ratios.std():.4f}")
    ratios_final = sub['ratio_final']
    in_range = ((ratios_final >= 0.5) & (ratios_final <= 0.9)).all()
    print(f"  ratio_final in [0.5, 0.9]: {in_range} "
          f"({[f'{v:.3f}' for v in ratios_final]})")

In [ ]:
# Plot relaxation ratios
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Relaxation Ratio at u=2', 'Relaxation Ratio at final u'],
    horizontal_spacing=0.12,
)

for Q, folders in Q_GROUPS.items():
    sub = relax_df[relax_df['folder'].isin(folders)].sort_values('L')
    if sub.empty:
        continue
    color = q_colors.get(Q, 'gray')

    fig.add_trace(go.Scatter(
        x=sub['L'], y=sub['ratio_u2'],
        mode='lines+markers+text',
        marker=dict(size=10, color=color),
        line=dict(color=color, width=2),
        text=[f'mb={mb}' for mb in sub['mb']],
        textposition='top center',
        name=f'Q={Q}', legendgroup=f'Q={Q}',
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=sub['L'], y=sub['ratio_final'],
        mode='lines+markers+text',
        marker=dict(size=10, color=color),
        line=dict(color=color, width=2),
        text=[f'mb={mb}' for mb in sub['mb']],
        textposition='top center',
        name=f'Q={Q}', legendgroup=f'Q={Q}', showlegend=False,
    ), row=1, col=2)

# Theory: 2/3 line
for col in [1, 2]:
    fig.add_hline(y=2/3, line_dash='dash', line_color='red', line_width=2,
                  annotation_text='2/3 theory', annotation_position='right',
                  row=1, col=col)
    fig.add_hrect(y0=0.5, y1=0.9, fillcolor='rgba(0,200,0,0.05)', line_width=0,
                  row=1, col=col)

fig.update_xaxes(title_text='L (msgs)', row=1, col=1)
fig.update_xaxes(title_text='L (msgs)', row=1, col=2)
fig.update_yaxes(title_text='I(u=2) / I(u=1)', row=1, col=1)
fig.update_yaxes(title_text='I_final / I_peak', row=1, col=2)

fig.update_layout(
    title_text='<b>Section 3: Relaxation Ratio</b> | Green band = [0.5, 0.9]',
    width=1200, height=450, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()

## Section 4: Participation Rate Scaling

**Execution intensity** = Q/L (shares per message) — how aggressively we trade.

- High Q/L: many shares per message (fast, concentrated execution)
- Low Q/L: few shares per message (slow, spread-out execution)

This section pools **ALL experiments** (not just Q-groups) and fits a power law: **I_peak ~ A × (Q/L)^δ**.

**Connection to 104's sqrt law:** In 104, beta ≈ 0.5 means `Impact ~ (Q/V)^0.5`. Here we use Q/L instead of Q/V (execution intensity instead of volume fraction). The exponent δ captures a related but different relationship — how peak impact scales with trading aggressiveness.

**Note:** We only have ~9 data points (one per experiment), so the power law fit is indicative, not definitive. The scatter also shows Q-group membership (colored) vs other experiments (gray diamonds) to highlight which points come from controlled comparisons.

In [ ]:
from scipy.stats import linregress

# Collect I_peak and execution parameters for ALL experiments
intensity_rows = []
for folder, c in all_curves.items():
    u = c['u_grid']
    mean = c['combined_mean']
    I_peak = float(np.interp(1.0, u, mean))

    intensity_rows.append({
        'folder': folder,
        'Q': c['Q'],
        'L': c['L'],
        'mb': c['mb'],
        'i': c['i'],
        'Q_over_L': c['Q'] / c['L'],
        'I_peak': I_peak,
    })

int_df = pd.DataFrame(intensity_rows)
int_df = int_df[int_df['I_peak'] > 0].copy()  # filter positive
int_df['log_QL'] = np.log(int_df['Q_over_L'])
int_df['log_Ipeak'] = np.log(int_df['I_peak'])

# Power law fit: I_peak = A * (Q/L)^delta
sl = linregress(int_df['log_QL'].values, int_df['log_Ipeak'].values)
delta = sl.slope
A = np.exp(sl.intercept)
r2 = sl.rvalue ** 2

print(f"Power law fit: I_peak = {A:.2f} * (Q/L)^{delta:.3f}")
print(f"R2 = {r2:.4f}, SE = {sl.stderr:.4f}, p = {sl.pvalue:.2e}")
print(f"\nAll experiments:")
print(int_df[['folder', 'Q', 'L', 'Q_over_L', 'I_peak']].to_string(index=False))

In [ ]:
fig = go.Figure()

# Color by Q-group membership
for Q, folders in Q_GROUPS.items():
    sub = int_df[int_df['folder'].isin(folders)]
    color = q_colors.get(Q, 'gray')
    fig.add_trace(go.Scatter(
        x=sub['Q_over_L'], y=sub['I_peak'],
        mode='markers+text',
        marker=dict(size=12, color=color, symbol='circle'),
        text=[f'mb={mb}' for mb in sub['mb']],
        textposition='top center',
        name=f'Q={Q} group',
    ))

# Other experiments (not in Q-groups)
q_group_folders = set(f for folders in Q_GROUPS.values() for f in folders)
other = int_df[~int_df['folder'].isin(q_group_folders)]
if not other.empty:
    fig.add_trace(go.Scatter(
        x=other['Q_over_L'], y=other['I_peak'],
        mode='markers+text',
        marker=dict(size=10, color='gray', symbol='diamond'),
        text=[f'i{i}mb{mb}' for i, mb in zip(other['i'], other['mb'])],
        textposition='top center',
        name='Other experiments',
    ))

# Fit line
x_fit = np.linspace(int_df['Q_over_L'].min() * 0.8, int_df['Q_over_L'].max() * 1.2, 100)
fig.add_trace(go.Scatter(
    x=x_fit, y=A * x_fit ** delta,
    mode='lines', line=dict(color='black', width=2, dash='dash'),
    name=f'Fit: {A:.1f} * (Q/L)^{delta:.2f} (R2={r2:.3f})',
))

fig.update_xaxes(title_text='Q/L (shares per message)', type='log')
fig.update_yaxes(title_text='I_peak = I(u=1) (price units)', type='log')

fig.update_layout(
    title_text=f'<b>Section 4: Peak Impact vs Execution Intensity</b> | delta={delta:.3f}',
    width=900, height=500, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)'),
)
fig.show()

## Section 5: Master Curve Collapse in u-Coordinates

**Idea:** If the model learns a *universal* impact dynamics, then after proper normalization, all curves from different experiments should **collapse onto one shape** — the "master curve".

**Normalization:** Each sample's impact is divided by `p_ref × σ_day` (pre-execution price × Parkinson daily volatility). This removes:
- Price level differences (GOOG at $900 vs $910 on different days)
- Volatility differences (high-vol day 2023-01-04 vs low-vol day 2023-01-05)

After this normalization, I_norm(u) is dimensionless and comparable across all days and experiments.

**What "collapse" means:**
- **Left plot**: Q-group curves — solid lines for Q=225, dashed for Q=150. If they overlap, the dynamics depend only on u, not on (Q, L) separately.
- **Right plot**: ALL experiments overlaid. If they collapse, the model has learned one universal impact shape.
- **CV metric**: Coefficient of variation across curves at each u-point. Low CV (< 0.3) = good collapse.

**Why this matters:** A universal master curve would mean the model captures a deep property of market microstructure — that price impact has a canonical shape regardless of execution parameters. This is a stronger claim than just "beta ≈ 0.5".

In [ ]:
# Plot sigma-normalized curves for all experiments
U_SHOW_NORM = 3.0

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Per Q-group (sigma-normalized)',
        'All experiments overlaid',
    ],
    horizontal_spacing=0.10,
)

# Left: per Q-group
for Q, folders in Q_GROUPS.items():
    for folder in folders:
        if folder not in all_curves_norm:
            continue
        c = all_curves_norm[folder]
        u = c['u_grid']
        mask = u <= U_SHOW_NORM
        u_show = u[mask]
        mean_show = c['combined_mean'][mask]

        line_color, _ = MB_COLORS[c['mb']]
        label = f'Q={Q} mb={c["mb"]} (L={c["L"]})'

        fig.add_trace(go.Scatter(
            x=u_show, y=mean_show,
            mode='lines', line=dict(color=line_color, width=2,
                                     dash='solid' if Q == 225 else 'dash'),
            name=label, legendgroup=label,
        ), row=1, col=1)

fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1.5, row=1, col=1)

# Right: all experiments (including non-Q-group)
colors_all = px.colors.qualitative.D3
sorted_folders = sorted(all_curves_norm.keys(),
                        key=lambda f: (parse_folder_params(f)[2], parse_folder_params(f)[0]))

for idx, folder in enumerate(sorted_folders):
    c = all_curves_norm[folder]
    u = c['u_grid']
    mask = u <= U_SHOW_NORM
    u_show = u[mask]
    mean_show = c['combined_mean'][mask]

    color = colors_all[idx % len(colors_all)]
    i_val, _, mb_val = parse_folder_params(folder)
    label = f'i{i_val}mb{mb_val}'

    fig.add_trace(go.Scatter(
        x=u_show, y=mean_show,
        mode='lines', line=dict(color=color, width=1.5),
        name=label, showlegend=True,
    ), row=1, col=2)

fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1.5, row=1, col=2)

fig.update_xaxes(title_text='u', row=1, col=1)
fig.update_xaxes(title_text='u', row=1, col=2)
fig.update_yaxes(title_text='I_norm(u) = I(u) / (p_ref * sigma_day)', row=1, col=1)
fig.update_yaxes(title_text='I_norm(u)', row=1, col=2)

fig.update_layout(
    title_text='<b>Section 5: Master Curve Collapse</b> | sigma-normalized impact',
    width=1500, height=500, template='plotly_white',
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)', font_size=9),
)
fig.show()

In [ ]:
# Quantify collapse: cross-curve variance at key u-points
u_eval_points = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5]

# Within each Q-group: how much do curves differ?
for Q, folders in Q_GROUPS.items():
    print(f"\n--- Q={Q} group ---")
    values_at_u = {u_pt: [] for u_pt in u_eval_points}
    labels = []
    for folder in folders:
        if folder not in all_curves_norm:
            continue
        c = all_curves_norm[folder]
        u = c['u_grid']
        mean = c['combined_mean']
        labels.append(f"mb={c['mb']}")
        for u_pt in u_eval_points:
            if u_pt <= u[-1]:
                values_at_u[u_pt].append(float(np.interp(u_pt, u, mean)))
            else:
                values_at_u[u_pt].append(np.nan)

    rows = []
    for u_pt in u_eval_points:
        vals = [v for v in values_at_u[u_pt] if not np.isnan(v)]
        if len(vals) >= 2:
            cv = np.std(vals) / abs(np.mean(vals)) if abs(np.mean(vals)) > 1e-12 else np.nan
            rows.append({'u': u_pt, 'mean': np.mean(vals), 'std': np.std(vals),
                         'cv': cv, 'values': vals})

    if rows:
        cv_df = pd.DataFrame(rows)
        print(cv_df[['u', 'mean', 'std', 'cv']].to_string(index=False))
        mean_cv = cv_df['cv'].mean()
        print(f"Mean CV across u-points: {mean_cv:.4f} (collapse if < 0.3)")

## Section 6: Summary Table + Comparison with 104

This section pulls together all the metrics computed above into one table and compares the volume-time dynamics (this notebook) with the static square-root law (104).

**Summary table** includes for each experiment:
- **I_peak** = I(u=1) — peak impact at end of execution
- **I_u2** = I(u=2) — impact after 1× execution duration of relaxation
- **I_final** — impact at the end of the observation window
- **relax_u2**, **relax_final** — relaxation ratios (what fraction of peak persists)
- **Q/L** — execution intensity (shares per message)

**Key questions answered:**
1. Does peak impact decrease with L at fixed Q? (Slower execution → less peak impact?)
2. Are relaxation ratios stable across experiments? (Universal decay dynamics?)
3. How does the power-law exponent δ from Section 4 compare with 104's β ≈ 0.5?
4. Does the 2×2 dashboard show a consistent picture across all metrics?

**Dashboard** (bottom): four key plots on one figure for a quick visual summary.

In [ ]:
# Build comprehensive summary
summary_rows = []

for folder in sorted(all_curves.keys(),
                      key=lambda f: (parse_folder_params(f)[2], parse_folder_params(f)[0])):
    c = all_curves[folder]
    u = c['u_grid']
    mean = c['combined_mean']

    I_peak = float(np.interp(1.0, u, mean))
    I_u2 = float(np.interp(2.0, u, mean)) if u[-1] >= 2.0 else np.nan
    I_final = float(mean[-1])

    # Q-group membership
    q_group = None
    for Q, folders in Q_GROUPS.items():
        if folder in folders:
            q_group = Q
            break

    row = {
        'folder': folder,
        'Q': c['Q'],
        'L': c['L'],
        'mb': c['mb'],
        'i': c['i'],
        'Q/L': c['Q'] / c['L'],
        'Q_group': q_group if q_group else '-',
        'I_peak': I_peak,
        'I_u2': I_u2,
        'I_final': I_final,
        'relax_u2': I_u2 / I_peak if not np.isnan(I_u2) and abs(I_peak) > 1e-12 else np.nan,
        'relax_final': I_final / I_peak if abs(I_peak) > 1e-12 else np.nan,
    }
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

print("=" * 130)
print("COMPREHENSIVE SUMMARY: Market Impact in Volume-Time")
print("=" * 130)
print(summary_df.to_string(index=False))

# Key findings
print("\n" + "=" * 80)
print("KEY FINDINGS")
print("=" * 80)

# 1. Peak decreases with L at fixed Q?
for Q, folders in Q_GROUPS.items():
    sub = summary_df[summary_df['folder'].isin(folders)].sort_values('L')
    if len(sub) > 1:
        peaks = sub['I_peak'].values
        monotonic = all(peaks[i] >= peaks[i+1] for i in range(len(peaks)-1))
        print(f"\n[Q={Q}] Peak decreasing with L: {monotonic}")
        print(f"  L = {list(sub['L'].values)}, I_peak = {[f'{v:.2f}' for v in peaks]}")

# 2. Relaxation ratios
valid_relax = summary_df['relax_final'].dropna()
print(f"\nRelaxation ratio (I_final/I_peak):")
print(f"  Range: [{valid_relax.min():.3f}, {valid_relax.max():.3f}]")
print(f"  Mean:  {valid_relax.mean():.3f}")
print(f"  All in [0.5, 0.9]: {((valid_relax >= 0.5) & (valid_relax <= 0.9)).all()}")

# 3. Power law exponent from Section 4
print(f"\nExecution intensity scaling: I_peak ~ (Q/L)^{delta:.3f} (R2={r2:.3f})")
print(f"  Compare with 104's sqrt law: beta ~ 0.5")

# 4. Comparison with 104
print("\n" + "-" * 80)
print("COMPARISON: 104 (static sqrt law) vs 105 (volume-time dynamics)")
print("-" * 80)
comparison = [
    ('What varies', 'Q (insertions)', 'L (execution speed at fixed Q)'),
    ('Time axis', 'None (iteration only)', 'Normalized u in [0, 1+gamma]'),
    ('Impact type', 'VWAP cumulative', 'Midprice instantaneous'),
    ('Key output', f'beta ~ 0.55', f'I(u) curve shape, delta ~ {delta:.2f}'),
    ('Question', 'Is beta = 0.5?', 'How does shape depend on L?'),
]
comp_df = pd.DataFrame(comparison, columns=['Aspect', '104 (sqrt law)', '105 (volume-time)'])
print(comp_df.to_string(index=False))

In [ ]:
# Summary dashboard: 2x2 key plots
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Volume-Time Curves (Q=225)',
        'I_peak vs Execution Speed L',
        'Relaxation Ratio',
        'Sigma-Normalized Collapse',
    ],
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

# (1,1) Volume-time curves for Q=225
for folder in Q_GROUPS.get(225, []):
    if folder not in all_curves:
        continue
    c = all_curves[folder]
    u = c['u_grid']
    mask = u <= 3.0
    line_color, _ = MB_COLORS[c['mb']]
    fig.add_trace(go.Scatter(
        x=u[mask], y=c['combined_mean'][mask],
        mode='lines', line=dict(color=line_color, width=2),
        name=f'mb={c["mb"]}', legendgroup=f'mb{c["mb"]}',
    ), row=1, col=1)
fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1, row=1, col=1)

# (1,2) I_peak vs L for Q-groups
for Q, folders in Q_GROUPS.items():
    sub = peak_df[peak_df['folder'].isin(folders)].sort_values('L')
    if sub.empty:
        continue
    color = q_colors.get(Q, 'gray')
    fig.add_trace(go.Scatter(
        x=sub['L'], y=sub['I_peak_u1'],
        mode='lines+markers',
        marker=dict(size=8, color=color),
        line=dict(color=color, width=2),
        name=f'Q={Q}', legendgroup=f'Q{Q}',
    ), row=1, col=2)

# (2,1) Relaxation ratios
for Q, folders in Q_GROUPS.items():
    sub = relax_df[relax_df['folder'].isin(folders)].sort_values('L')
    if sub.empty:
        continue
    color = q_colors.get(Q, 'gray')
    fig.add_trace(go.Scatter(
        x=sub['L'], y=sub['ratio_final'],
        mode='lines+markers',
        marker=dict(size=8, color=color),
        line=dict(color=color, width=2),
        name=f'Q={Q}', legendgroup=f'Q{Q}', showlegend=False,
    ), row=2, col=1)
fig.add_hline(y=2/3, line_dash='dash', line_color='red', line_width=1.5, row=2, col=1)

# (2,2) Sigma-normalized curves (all)
for idx, folder in enumerate(sorted_folders):
    if folder not in all_curves_norm:
        continue
    c = all_curves_norm[folder]
    u = c['u_grid']
    mask = u <= 3.0
    color = colors_all[idx % len(colors_all)]
    i_val, _, mb_val = parse_folder_params(folder)
    fig.add_trace(go.Scatter(
        x=u[mask], y=c['combined_mean'][mask],
        mode='lines', line=dict(color=color, width=1.5),
        name=f'i{i_val}mb{mb_val}', showlegend=False,
    ), row=2, col=2)
fig.add_vline(x=1.0, line_dash='dash', line_color='gray', line_width=1, row=2, col=2)

fig.update_xaxes(title_text='u', row=1, col=1)
fig.update_yaxes(title_text='I(u)', row=1, col=1)
fig.update_xaxes(title_text='L (msgs)', row=1, col=2)
fig.update_yaxes(title_text='I_peak', row=1, col=2)
fig.update_xaxes(title_text='L (msgs)', row=2, col=1)
fig.update_yaxes(title_text='I_final / I_peak', row=2, col=1)
fig.update_xaxes(title_text='u', row=2, col=2)
fig.update_yaxes(title_text='I_norm(u)', row=2, col=2)

fig.update_layout(
    title_text='<b>Section 6: Summary Dashboard</b>',
    width=1400, height=800, template='plotly_white',
    showlegend=True,
    legend=dict(x=0.01, y=0.99, bgcolor='rgba(255,255,255,0.8)', font_size=9),
)
fig.show()